# Apnoeic oxygen desaturation simulator — editable notebook

This notebook reproduces the core deterministic model, the external validation figures, the dynamic-shunt supplement, a one-at-a-time sensitivity check, and a lightweight Monte Carlo analysis.

**Educational/research use only.**  
This model does **not** predict individual patient risk, does **not** define safe apnoea time, and must **not** guide airway or oxygenation management.

Suggested workflow:

1. Run all cells once.
2. Modify the editable parameter blocks.
3. Re-run the plot/validation cells.
4. Export figures at 300 dpi for manuscript use.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from scipy.integrate import solve_ivp

plt.rcParams["figure.dpi"] = 120

P_BARO   = 760.0
P_H2O    = 47.0
HUFNER   = 1.34
ALPHA_O2 = 0.003

def severinghaus_sao2(po2_mmHg):
    """Severinghaus-like Hb-O2 dissociation curve. PO2 -> SaO2 fraction."""
    po2 = np.maximum(po2_mmHg, 1e-6)
    return 1.0 / ((23400.0 / (po2**3 + 150.0 * po2)) + 1.0)

def severinghaus_po2(sao2_frac):
    """Numerical inverse SaO2 -> PO2."""
    scalar_in = np.isscalar(sao2_frac) or (np.ndim(sao2_frac) == 0)
    sao2 = np.clip(np.atleast_1d(sao2_frac).astype(float), 1e-6, 0.999999)
    lo = np.full_like(sao2, 1.0)
    hi = np.full_like(sao2, 700.0)
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        f = severinghaus_sao2(mid)
        hi = np.where(f > sao2, mid, hi)
        lo = np.where(f > sao2, lo, mid)
    out = 0.5 * (lo + hi)
    return float(out[0]) if scalar_in else out

def o2_content(hb_gdl, sao2_frac, po2_mmHg):
    return HUFNER * hb_gdl * sao2_frac + ALPHA_O2 * po2_mmHg

In [ ]:
@dataclass
class Population:
    nome: str
    eta_anni: float
    peso_kg: float
    frc_ml: float
    vo2_ml_min: float
    hb_gdl: float
    co_dl_min: float
    shunt: float
    fao2_iniz: float
    airway_pervio: bool
    f_aw_o2: float
    r_apnea: float = 0.06
    vol_sangue_dl: float = None
    dshunt_max: float = 0.0
    tau_atel_s: float = 90.0
    atel_patent_factor: float = 0.3
    f_trapped: float = 0.0

    def __post_init__(self):
        if self.vol_sangue_dl is None:
            self.vol_sangue_dl = 0.75 * self.peso_kg  # 75 mL/kg = 0.75 dL/kg

def vo2_riferimento_ml_min(eta_anni, peso_kg):
    """Allometric VO2, calibrated to adult 70 kg -> 250 mL/min."""
    K_VO2 = 250.0 / (70.0 ** 0.75)
    return K_VO2 * peso_kg ** 0.75

def frc_riferimento_ml(eta_anni, peso_kg):
    """Anaesthetised FRC reference: smooth age-dependent mL/kg curve."""
    k_frc = 30.0 - 13.0 * np.exp(-eta_anni / 2.5)
    return k_frc * peso_kg

def co_riferimento_dl_min(eta_anni, peso_kg):
    """Allometric cardiac output, calibrated to adult 70 kg -> 5 L/min."""
    K_CO = 5000.0 / (70.0 ** 0.75)
    return (K_CO * peso_kg ** 0.75) / 100.0

def hb_riferimento_gdl(eta_anni):
    eta_nodi = [0.0, 0.05, 0.25, 0.5, 1.0, 5.0, 12.0, 18.0]
    hb_nodi  = [17.0, 15.0, 11.0, 11.5, 12.0, 12.5, 13.5, 14.5]
    return float(np.interp(eta_anni, eta_nodi, hb_nodi))

F0_CC = 0.22
AGE_SCALE_CC = 4.0

def f_trapped_cc(eta_anni, f0=F0_CC, scale=AGE_SCALE_CC):
    """Age-dependent trapped FRC fraction used by the closing-capacity sensitivity module."""
    return f0 * np.exp(-eta_anni / scale)

In [ ]:
def shunt_at(t_s, p: Population):
    factor = p.atel_patent_factor if p.airway_pervio else 1.0
    return p.shunt + p.dshunt_max * (1.0 - np.exp(-t_s / p.tau_atel_s)) * factor

def apnea_odes(t, y, p: Population):
    A_L, Cv = y
    A_L = max(A_L, 1e-6)
    Cv  = max(Cv, 1e-6)

    frc_eff = p.frc_ml * (1.0 - p.f_trapped)
    fao2 = min(max(A_L / frc_eff, 0.0), 1.0)
    pao2 = fao2 * (P_BARO - P_H2O)

    scc = severinghaus_sao2(pao2)
    ccc = o2_content(p.hb_gdl, scc, pao2)

    sv = np.clip(Cv / (HUFNER * p.hb_gdl), 1e-6, 1.0)
    pv = severinghaus_po2(sv)
    cv = o2_content(p.hb_gdl, sv, pv)

    s_shunt = shunt_at(t, p)
    ca = (1.0 - s_shunt) * ccc + s_shunt * cv

    j_up = p.co_dl_min * (ca - cv)
    j_up = max(j_up, 0.0)

    vco2_alv = p.r_apnea * p.vo2_ml_min
    j_net = max(j_up - vco2_alv, 0.0)

    o2_in_aw = j_net * p.f_aw_o2 if p.airway_pervio else 0.0

    dA_L = o2_in_aw - j_up
    dCv = (j_up - p.vo2_ml_min) / p.vol_sangue_dl

    return [dA_L, dCv]

def simula(p: Population, t_max_s=1800.0, dt_s=1.0, rtol=1e-7, atol=1e-9, max_step=2.0):
    frc_eff = p.frc_ml * (1.0 - p.f_trapped)

    A_L0 = p.fao2_iniz * frc_eff
    pao2_0 = p.fao2_iniz * (P_BARO - P_H2O)
    sa0 = severinghaus_sao2(pao2_0)
    ca0 = o2_content(p.hb_gdl, sa0, pao2_0)
    cv0 = ca0 - p.vo2_ml_min / p.co_dl_min

    t_eval = np.arange(0.0, t_max_s + dt_s, dt_s)
    sol = solve_ivp(
        lambda t, y: [d / 60.0 for d in apnea_odes(t, y, p)],
        [0.0, t_max_s],
        [A_L0, cv0],
        t_eval=t_eval,
        method="LSODA",
        rtol=rtol,
        atol=atol,
        max_step=max_step
    )

    A_L, Cv = sol.y
    fao2 = np.clip(A_L / frc_eff, 0, 1)
    pao2 = fao2 * (P_BARO - P_H2O)
    scc  = severinghaus_sao2(pao2)
    ccc  = o2_content(p.hb_gdl, scc, pao2)
    sv   = np.clip(Cv / (HUFNER * p.hb_gdl), 1e-6, 1.0)
    pv   = severinghaus_po2(sv)
    cv   = o2_content(p.hb_gdl, sv, pv)
    s_sh = shunt_at(sol.t, p)
    ca   = (1 - s_sh) * ccc + s_sh * cv
    sa   = np.clip((ca - ALPHA_O2 * pao2) / (HUFNER * p.hb_gdl), 1e-6, 1.0)

    return sol.t, sa * 100.0, pao2

def tempo_a_soglia(t, spo2, soglia=90.0):
    sotto = np.where(spo2 <= soglia)[0]
    if len(sotto) == 0:
        return None
    i = sotto[0]
    if i == 0:
        return 0.0
    t0, t1 = t[i-1], t[i]
    s0, s1 = spo2[i-1], spo2[i]
    return float(t0 + (soglia - s0) * (t1 - t0) / (s1 - s0))

def metriche_agreement(osservati, predetti):
    o = np.array(osservati, float)
    pr = np.array(predetti, float)
    diff = pr - o
    bias = np.mean(diff)
    sd   = np.std(diff, ddof=1) if len(diff) > 1 else 0.0
    return {
        "n": len(o),
        "bias_s": bias,
        "rmse_s": float(np.sqrt(np.mean(diff**2))),
        "mae_s":  float(np.mean(np.abs(diff))),
        "loa_inf_s": bias - 1.96 * sd,
        "loa_sup_s": bias + 1.96 * sd,
    }

In [ ]:
TAU_ATEL = 90.0
DSHUNT_LETTERATURA = 0.10

ETA_EN = {
    "0-6 mesi": "0-6 mo", "7-23 mesi": "7-23 mo", "2-5 anni": "2-5 yr",
    "6-10 anni": "6-10 yr", "11-18 anni": "11-18 yr", "adulto sano": "adult"
}

OSSERVATI_90 = {
    "0-6 mesi":     96.5,
    "7-23 mesi":   118.5,
    "2-5 anni":    160.4,
    "6-10 anni":   214.9,
    "11-18 anni":  382.4,
    "adulto sano": 364.0,
}
OSSERVATI_90_SD = {
    "0-6 mesi": 12.7, "7-23 mesi": 9.0, "2-5 anni": 30.7,
    "6-10 anni": 34.9, "11-18 anni": 79.9, "adulto sano": None,
}

XUE_1996 = {
    # nome:        (eta, peso, Hb,   T99,   T95,   T90)
    "Xue 3mo-1a":  (0.8,  7.5, 12.4,  95.0, 110.2, 118.5),
    "Xue 1-3a":    (2.0, 12.6, 13.1, 127.5, 154.8, 168.7),
    "Xue 3-12a":   (7.6, 24.7, 13.8, 210.3, 231.8, 248.0),
}

def costruisci(nome, eta, peso, fao2=0.90, airway_pervio=False, f_aw_o2=0.0,
               shunt=0.05, dshunt_max=0.0, use_cc=False):
    return Population(
        nome=nome,
        eta_anni=eta,
        peso_kg=peso,
        frc_ml=frc_riferimento_ml(eta, peso),
        vo2_ml_min=vo2_riferimento_ml_min(eta, peso),
        hb_gdl=hb_riferimento_gdl(eta),
        co_dl_min=co_riferimento_dl_min(eta, peso),
        shunt=shunt,
        fao2_iniz=fao2,
        airway_pervio=airway_pervio,
        f_aw_o2=f_aw_o2,
        dshunt_max=dshunt_max,
        tau_atel_s=TAU_ATEL,
        f_trapped=(f_trapped_cc(eta) if use_cc else 0.0),
    )

def build_patel(dshunt_max=0.0, use_cc=False):
    return [
        costruisci("0-6 mesi",   0.25,  6.0, dshunt_max=dshunt_max, use_cc=use_cc),
        costruisci("7-23 mesi",  1.25, 10.5, dshunt_max=dshunt_max, use_cc=use_cc),
        costruisci("2-5 anni",   3.5,  15.0, dshunt_max=dshunt_max, use_cc=use_cc),
        costruisci("6-10 anni",  8.0,  26.0, dshunt_max=dshunt_max, use_cc=use_cc),
        costruisci("11-18 anni", 14.5, 50.0, dshunt_max=dshunt_max, use_cc=use_cc),
    ]

def build_adulto(dshunt_max=0.0):
    return Population(
        nome="adulto sano", eta_anni=35, peso_kg=70,
        frc_ml=2300, vo2_ml_min=250, hb_gdl=15, co_dl_min=50,
        shunt=0.05, fao2_iniz=0.90, airway_pervio=False, f_aw_o2=0.0,
        dshunt_max=dshunt_max, tau_atel_s=TAU_ATEL
    )

def build_thrive(dshunt_max=0.0):
    return Population(
        nome="adulto THRIVE", eta_anni=45, peso_kg=80,
        frc_ml=2400, vo2_ml_min=250, hb_gdl=14, co_dl_min=55,
        shunt=0.08, fao2_iniz=0.90, airway_pervio=True, f_aw_o2=1.0,
        dshunt_max=dshunt_max, tau_atel_s=TAU_ATEL
    )

## Editable single-scenario simulation

Modify the parameters below and re-run the next cell.

In [ ]:
# EDIT HERE
single = costruisci(
    nome="custom child",
    eta=3.5,
    peso=15.0,
    fao2=0.90,
    airway_pervio=False,
    f_aw_o2=0.0,
    shunt=0.05,
    dshunt_max=0.0,
    use_cc=True
)

t, spo2, pao2 = simula(single, t_max_s=1200)
t90 = tempo_a_soglia(t, spo2, 90.0)

print(f"{single.nome}: T90 = {t90:.1f} s" if t90 else f"{single.nome}: no T90 within window")
print(f"FRC={single.frc_ml:.1f} mL, effective FRC={single.frc_ml*(1-single.f_trapped):.1f} mL")
print(f"VO2={single.vo2_ml_min:.1f} mL/min, Hb={single.hb_gdl:.1f} g/dL, CO={single.co_dl_min:.1f} dL/min")

plt.figure(figsize=(7, 4.5))
plt.plot(t/60, spo2, lw=2)
plt.axhline(90, ls="--", color="gray")
if t90:
    plt.axvline(t90/60, ls=":", color="gray")
    plt.scatter([t90/60], [90], zorder=5)
plt.xlabel("Apnoea time (min)")
plt.ylabel("SpO$_2$ (%)")
plt.title("Single-scenario simulation")
plt.ylim(60, 101)
plt.grid(alpha=0.3)
plt.show()

## Figure 1 — age-dependent gradient and THRIVE limiting case

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

pops = build_patel(0.0, use_cc=False) + [build_adulto(0.0)]
for p in pops:
    t, spo2, _ = simula(p, t_max_s=1800.0)
    ax1.plot(t / 60.0, spo2, lw=2.0, label=ETA_EN.get(p.nome, p.nome))

ax1.axhline(90, ls="--", color="gray", lw=1)
ax1.set_xlim(0, 12)
ax1.set_ylim(60, 101)
ax1.set_xlabel("Apnoea time (min)")
ax1.set_ylabel("SpO$_2$ (%)")
ax1.set_title("Age-dependent gradient (closed airway)")
ax1.legend(fontsize=8, loc="lower left")
ax1.grid(alpha=0.3)

tA, sA, _ = simula(build_adulto(0.0), t_max_s=1800.0)
tT, sT, _ = simula(build_thrive(0.0), t_max_s=1800.0)
ax2.plot(tA / 60.0, sA, lw=2.2, label="closed airway (adult)")
ax2.plot(tT / 60.0, sT, lw=2.2, label="apnoeic oxygenation (THRIVE)")
ax2.axhline(90, ls="--", color="gray", lw=1)
ax2.set_xlim(0, 20)
ax2.set_ylim(60, 101)
ax2.set_xlabel("Apnoea time (min)")
ax2.set_ylabel("SpO$_2$ (%)")
ax2.set_title("Limiting case: apnoeic oxygenation (THRIVE) vs closed airway")
ax2.legend(fontsize=8, loc="lower left")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("figure1_age_gradient_thrive.png", dpi=300, bbox_inches="tight")
plt.show()

## External validation helpers

In [ ]:
def esegui_patel(cohort, label=""):
    risultati, curve, oss_l, pred_l = {}, {}, [], []
    print(f"\nPatel 1994 [{label}]")
    print(f"{'group':14s} {'pred':>8s} {'obs':>8s} {'error':>8s} {'error%':>8s}")
    for p in cohort:
        t, spo2, _ = simula(p, t_max_s=1800.0)
        t90 = tempo_a_soglia(t, spo2, 90.0)
        risultati[p.nome] = t90
        curve[p.nome] = (t, spo2)
        obs = OSSERVATI_90.get(p.nome)
        if obs is not None and t90 is not None:
            err = t90 - obs
            oss_l.append(obs)
            pred_l.append(t90)
            print(f"{ETA_EN.get(p.nome,p.nome):14s} {t90:7.1f}s {obs:7.1f}s {err:+7.1f}s {100*err/obs:+7.1f}%")
    return risultati, curve, oss_l, pred_l

def cross_validazione_xue(dshunt_max=0.0, label="", use_cc=False):
    oss_l, pred_l = [], []
    print(f"\nXue 1996 [{label}]")
    print(f"{'group':14s} {'pred':>8s} {'obs':>8s} {'error':>8s} {'error%':>8s}")
    for nome, (eta, peso, hb, t99, t95, t90) in XUE_1996.items():
        p = Population(
            nome=nome, eta_anni=eta, peso_kg=peso,
            frc_ml=frc_riferimento_ml(eta, peso),
            vo2_ml_min=vo2_riferimento_ml_min(eta, peso),
            hb_gdl=hb,
            co_dl_min=co_riferimento_dl_min(eta, peso),
            shunt=0.05,
            fao2_iniz=0.90,
            airway_pervio=False,
            f_aw_o2=0.0,
            dshunt_max=dshunt_max,
            tau_atel_s=TAU_ATEL,
            f_trapped=(f_trapped_cc(eta) if use_cc else 0.0),
        )
        t, spo2, _ = simula(p, t_max_s=1800.0)
        pred = tempo_a_soglia(t, spo2, 90.0)
        err = pred - t90
        oss_l.append(t90)
        pred_l.append(pred)
        print(f"{nome:14s} {pred:7.1f}s {t90:7.1f}s {err:+7.1f}s {100*err/t90:+7.1f}%")
    return oss_l, pred_l

def print_metrics(name, obs, pred):
    m = metriche_agreement(obs, pred)
    mape = np.mean(np.abs((np.array(pred) - np.array(obs)) / np.array(obs))) * 100
    r = np.corrcoef(obs, pred)[0, 1] if len(obs) > 1 else np.nan
    print(f"{name:25s} bias={m['bias_s']:+.1f}s  RMSE={m['rmse_s']:.1f}s  MAPE={mape:.1f}%  r={r:.3f}")
    return {"name": name, **m, "mape": mape, "r": r}

## Figure 2 — external validation and closing-capacity module

In [ ]:
res_b, cur_b, ob, pb = esegui_patel(build_patel(0.0, use_cc=False), "baseline")
oxb, pxb = cross_validazione_xue(0.0, "baseline", use_cc=False)

res_c, cur_c, oc, pc = esegui_patel(build_patel(0.0, use_cc=True), "closing capacity")
oxc, pxc = cross_validazione_xue(0.0, "closing capacity", use_cc=True)

print("\nAgreement metrics")
metrics = [
    print_metrics("Patel baseline", ob, pb),
    print_metrics("Patel closing capacity", oc, pc),
    print_metrics("Xue baseline", oxb, pxb),
    print_metrics("Xue closing capacity", oxc, pxc),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

colori = {"0-6 mesi": "C0", "7-23 mesi": "C1", "2-5 anni": "C2",
          "6-10 anni": "C3", "11-18 anni": "C4"}

for nome, c in colori.items():
    tb, sb = cur_b[nome]
    tc, sc = cur_c[nome]
    ax1.plot(tb / 60.0, sb, ls="--", lw=1.3, color=c, alpha=0.55)
    ax1.plot(tc / 60.0, sc, ls="-", lw=2.0, color=c, label=ETA_EN[nome])
    obs = OSSERVATI_90[nome]
    sd = OSSERVATI_90_SD[nome]
    ax1.errorbar(obs/60.0, 90, xerr=sd/60.0, fmt="o", color=c, ms=7, capsize=3, mec="k", mew=0.7)

ax1.axhline(90, ls=":", color="gray", lw=1)
ax1.set_xlim(0, 8)
ax1.set_ylim(60, 101)
ax1.set_xlabel("Apnoea time (min)")
ax1.set_ylabel("SpO$_2$ (%)")
ax1.set_title("Patel: baseline (- -) vs closing capacity (—) + obs. ($\\bullet$)")
ax1.legend(fontsize=8, loc="lower left")
ax1.grid(alpha=0.3)

lim = 520
ax2.plot([0, lim], [0, lim], "k--", lw=1, label="identity")
ax2.scatter(ob, pb, marker="o", facecolor="none", edgecolor="C3", s=70, label="Patel baseline")
ax2.scatter(oc, pc, marker="o", color="C3", s=70, edgecolor="k", label="Patel closing cap.")
ax2.scatter(oxb, pxb, marker="s", facecolor="none", edgecolor="C0", s=70, label="Xue baseline")
ax2.scatter(oxc, pxc, marker="s", color="C0", s=70, edgecolor="k", label="Xue closing cap.")
ax2.set_xlim(0, lim)
ax2.set_ylim(0, lim)
ax2.set_xlabel("Observed T$_{90}$ (s)")
ax2.set_ylabel("Predicted T$_{90}$ (s)")
ax2.set_title("Agreement: the module moves points toward identity")
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)
ax2.set_aspect("equal")

plt.tight_layout()
plt.savefig("figure2_validation_closing_capacity.png", dpi=300, bbox_inches="tight")
plt.show()

## Supplementary figure — fixed shunt versus dynamic shunt

In [ ]:
res_fixed, cur_fixed, of, pf = esegui_patel(build_patel(0.0, use_cc=False), "fixed shunt")
oxf, pxf = cross_validazione_xue(0.0, "fixed shunt", use_cc=False)

res_dyn, cur_dyn, od, pd = esegui_patel(build_patel(DSHUNT_LETTERATURA, use_cc=False), "dynamic shunt")
oxd, pxd = cross_validazione_xue(DSHUNT_LETTERATURA, "dynamic shunt", use_cc=False)

print("\nDynamic shunt metrics")
_ = [
    print_metrics("Patel fixed shunt", of, pf),
    print_metrics("Patel dynamic shunt", od, pd),
    print_metrics("Xue fixed shunt", oxf, pxf),
    print_metrics("Xue dynamic shunt", oxd, pxd),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

for nome, c in colori.items():
    tf, sf = cur_fixed[nome]
    td, sd = cur_dyn[nome]
    ax1.plot(tf / 60.0, sf, ls="--", lw=1.3, color=c, alpha=0.55)
    ax1.plot(td / 60.0, sd, ls="-", lw=2.0, color=c, label=ETA_EN[nome])
    obs = OSSERVATI_90[nome]
    sdobs = OSSERVATI_90_SD[nome]
    ax1.errorbar(obs/60.0, 90, xerr=sdobs/60.0, fmt="o", color=c, ms=7, capsize=3, mec="k", mew=0.7)

ax1.axhline(90, ls=":", color="gray", lw=1)
ax1.set_xlim(0, 8)
ax1.set_ylim(60, 101)
ax1.set_xlabel("Apnoea time (min)")
ax1.set_ylabel("SpO$_2$ (%)")
ax1.set_title("Suppl.: fixed shunt (- -) vs dynamic shunt (—) + obs. ($\\bullet$)")
ax1.legend(fontsize=8, loc="lower left")
ax1.grid(alpha=0.3)

lim = 520
ax2.plot([0, lim], [0, lim], "k--", lw=1, label="identity")
ax2.scatter(of, pf, marker="o", facecolor="none", edgecolor="C3", s=70, label="Patel fixed shunt")
ax2.scatter(od, pd, marker="o", color="C3", s=70, edgecolor="k", label="Patel dynamic shunt")
ax2.scatter(oxf, pxf, marker="s", facecolor="none", edgecolor="C0", s=70, label="Xue fixed shunt")
ax2.scatter(oxd, pxd, marker="s", color="C0", s=70, edgecolor="k", label="Xue dynamic shunt")
ax2.set_xlim(0, lim)
ax2.set_ylim(0, lim)
ax2.set_xlabel("Observed T$_{90}$ (s)")
ax2.set_ylabel("Predicted T$_{90}$ (s)")
ax2.set_title("Agreement unchanged: rising shunt does not shift onset")
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)
ax2.set_aspect("equal")

plt.tight_layout()
plt.savefig("figureS1_dynamic_shunt.png", dpi=300, bbox_inches="tight")
plt.show()

## A-priori Table 1 generated from code

Use this cell to avoid manual mismatch between manuscript and code.

In [ ]:
rows = []
for p in build_patel(0.0, use_cc=True):
    rows.append([
        ETA_EN[p.nome],
        p.frc_ml,
        p.vo2_ml_min,
        p.hb_gdl,
        100 * p.f_trapped
    ])

print(f"{'Age band':10s} {'FRC (mL)':>10s} {'VO2 (mL/min)':>14s} {'Hb':>7s} {'trapped FRC':>14s}")
for row in rows:
    print(f"{row[0]:10s} {row[1]:10.0f} {row[2]:14.1f} {row[3]:7.1f} {row[4]:13.1f}%")

## One-at-a-time sensitivity analysis

This is a simple reproducible implementation for the manuscript statement that T90 is mainly driven by effective FRC and VO₂.

In [ ]:
def copy_population(p, **kwargs):
    data = p.__dict__.copy()
    data.update(kwargs)
    return Population(**data)

def sensitivity_one_at_a_time(base_pop, perturb=0.20):
    base_t, base_spo2, _ = simula(base_pop, t_max_s=1800)
    base_t90 = tempo_a_soglia(base_t, base_spo2, 90.0)
    tests = {}

    p_frc = copy_population(base_pop, frc_ml=base_pop.frc_ml * (1 + perturb))
    tests["FRC +20%"] = tempo_a_soglia(*simula(p_frc, t_max_s=1800)[:2], soglia=90.0)

    p_vo2 = copy_population(base_pop, vo2_ml_min=base_pop.vo2_ml_min * (1 + perturb))
    tests["VO2 +20%"] = tempo_a_soglia(*simula(p_vo2, t_max_s=1800)[:2], soglia=90.0)

    p_vol = copy_population(base_pop, vol_sangue_dl=base_pop.vol_sangue_dl * (1 + perturb))
    tests["blood-tissue store +20%"] = tempo_a_soglia(*simula(p_vol, t_max_s=1800)[:2], soglia=90.0)

    p_fao2 = copy_population(base_pop, fao2_iniz=min(base_pop.fao2_iniz * (1 + perturb), 0.96))
    tests["FAO2 +20% clipped"] = tempo_a_soglia(*simula(p_fao2, t_max_s=1800)[:2], soglia=90.0)

    p_shunt = copy_population(base_pop, shunt=min(base_pop.shunt * (1 + perturb), 0.99))
    tests["shunt +20%"] = tempo_a_soglia(*simula(p_shunt, t_max_s=1800)[:2], soglia=90.0)

    print(f"Base population: {base_pop.nome}; baseline T90={base_t90:.1f} s")
    print(f"{'parameter':28s} {'T90':>10s} {'change':>10s} {'elasticity':>12s}")
    out = []
    for k, v in tests.items():
        change = (v - base_t90) / base_t90
        elasticity = change / perturb
        out.append((k, v, change, elasticity))
        print(f"{k:28s} {v:9.1f}s {100*change:+9.1f}% {elasticity:+11.2f}")
    return base_t90, out

base_pop = build_patel(0.0, use_cc=False)[2]
base_t90, sens = sensitivity_one_at_a_time(base_pop, perturb=0.20)

## Lightweight Monte Carlo analysis

This cell is intentionally lighter than a full production run. Increase `N_INDIVIDUAL`, `N_REP`, and `T_MAX_MC` for final analyses.

In [ ]:
CV_FRC, CV_VO2, CV_CO, CV_CC = 0.22, 0.18, 0.18, 0.40
SD_HB, SD_FAO2, SD_SHUNT = 1.2, 0.03, 0.03
R_APNEA = 0.06

def _logn(cv, rng, n):
    sigma = np.sqrt(np.log(1.0 + cv**2))
    return rng.lognormal(mean=-0.5 * sigma**2, sigma=sigma, size=n)

def _deriv_mc(A_L, Cv, P):
    frc_eff = P["frc"] * (1.0 - P["ftr"])
    fao2 = np.clip(A_L / frc_eff, 0.0, 1.0)
    pao2 = fao2 * (P_BARO - P_H2O)
    scc = severinghaus_sao2(pao2)
    ccc = HUFNER * P["hb"] * scc + ALPHA_O2 * pao2
    # Fast Monte Carlo approximation: Cv is already mixed-venous O2 content.
    # The dissolved venous O2 term is negligible for this sensitivity-level simulation.
    cv = np.maximum(Cv, 1e-6)
    ca = (1.0 - P["shunt"]) * ccc + P["shunt"] * cv
    j_up = np.maximum(P["co"] * (ca - cv), 0.0)
    j_net = np.maximum(j_up - R_APNEA * P["vo2"], 0.0)
    o2_in = j_net * P["f_aw_o2"] * P["patent"]
    return o2_in - j_up, (j_up - P["vo2"]) / P["vol_sangue"]

def _sao2_mc(A_L, Cv, P):
    frc_eff = P["frc"] * (1.0 - P["ftr"])
    fao2 = np.clip(A_L / frc_eff, 0.0, 1.0)
    pao2 = fao2 * (P_BARO - P_H2O)
    scc = severinghaus_sao2(pao2)
    ccc = HUFNER * P["hb"] * scc + ALPHA_O2 * pao2
    # Fast Monte Carlo approximation: Cv is already mixed-venous O2 content.
    # The dissolved venous O2 term is negligible for this sensitivity-level simulation.
    cv = np.maximum(Cv, 1e-6)
    ca = (1.0 - P["shunt"]) * ccc + P["shunt"] * cv
    return np.clip((ca - ALPHA_O2 * pao2) / (HUFNER * P["hb"]), 1e-6, 1.0) * 100.0

def monte_carlo(eta, peso, hb_mean, n=500, t_max_s=1800.0, dt=2.0, seed=0, use_cc=True, fao2_mean=0.90, drop_nan=True):
    rng = np.random.default_rng(seed)
    frc0 = frc_riferimento_ml(eta, peso)
    vo20 = vo2_riferimento_ml_min(eta, peso)
    co0 = co_riferimento_dl_min(eta, peso)
    ftr0 = f_trapped_cc(eta) if use_cc else 0.0

    P = {
        "frc": frc0 * _logn(CV_FRC, rng, n),
        "vo2": vo20 * _logn(CV_VO2, rng, n),
        "co":  co0 * _logn(CV_CO, rng, n),
        "hb":  np.clip(rng.normal(hb_mean, SD_HB, n), 7.0, 20.0),
        "shunt": np.clip(rng.normal(0.05, SD_SHUNT, n), 0.0, 0.30),
        "ftr": np.clip(ftr0 * _logn(CV_CC, rng, n), 0.0, 0.40),
        "f_aw_o2": np.zeros(n),
        "patent": np.zeros(n),
    }
    P["vol_sangue"] = 0.75 * peso * np.ones(n)

    fao2 = np.clip(rng.normal(fao2_mean, SD_FAO2, n), 0.80, 0.96)
    A_L = fao2 * P["frc"] * (1.0 - P["ftr"])
    pao2_0 = fao2 * (P_BARO - P_H2O)
    sa0 = severinghaus_sao2(pao2_0)
    ca0 = HUFNER * P["hb"] * sa0 + ALPHA_O2 * pao2_0
    Cv = ca0 - P["vo2"] / P["co"]

    nstep = int(t_max_s / dt)
    t90 = np.full(n, np.nan)
    prev_sa = _sao2_mc(A_L, Cv, P)
    h = dt / 60.0

    for k in range(1, nstep + 1):
        k1a, k1c = _deriv_mc(A_L, Cv, P)
        k2a, k2c = _deriv_mc(A_L + 0.5*h*k1a, Cv + 0.5*h*k1c, P)
        k3a, k3c = _deriv_mc(A_L + 0.5*h*k2a, Cv + 0.5*h*k2c, P)
        k4a, k4c = _deriv_mc(A_L + h*k3a, Cv + h*k3c, P)
        A_L = np.maximum(A_L + (h/6)*(k1a+2*k2a+2*k3a+k4a), 1e-6)
        Cv  = np.maximum(Cv + (h/6)*(k1c+2*k2c+2*k3c+k4c), 1e-6)
        sa = _sao2_mc(A_L, Cv, P)

        cross = (prev_sa > 90.0) & (sa <= 90.0) & np.isnan(t90)
        if cross.any():
            frac = (prev_sa[cross] - 90.0) / (prev_sa[cross] - sa[cross])
            t90[cross] = (k - 1 + frac) * dt
        prev_sa = sa

    return t90[~np.isnan(t90)] if drop_nan else t90

def medie_di_gruppo(eta, peso, hb_mean, n_group, n_rep=200, t_max_s=1800.0, seed=0, use_cc=True):
    n_tot = n_rep * n_group
    t90 = monte_carlo(eta, peso, hb_mean, n=n_tot, t_max_s=t_max_s, seed=seed, use_cc=use_cc, drop_nan=False)
    return np.nanmean(t90.reshape(n_rep, n_group), axis=1)

COORTE = {
    "0-6 mesi":   (0.25,  6.0, None,  96.5, 12.7, 10),
    "7-23 mesi":  (1.25, 10.5, None, 118.5,  9.0, 10),
    "2-5 anni":   (3.5,  15.0, None, 160.4, 30.7, 10),
    "6-10 anni":  (8.0,  26.0, None, 214.9, 34.9, 10),
    "11-18 anni": (14.5, 50.0, None, 382.4, 79.9, 10),
}

N_INDIVIDUAL = 400
N_REP = 100
T_MAX_MC = 1800.0
DT_MC = 3.0

fasce, obs_mean, obs_sd = [], [], []
gm_lo, gm_hi, gm_mean = [], [], []
camp = []

for nome, (eta, peso, hbm, o, sd, ng) in COORTE.items():
    hb = hb_riferimento_gdl(eta) if hbm is None else hbm
    gmeans = medie_di_gruppo(eta, peso, hb, ng, n_rep=N_REP, seed=42, t_max_s=T_MAX_MC, use_cc=True)
    individual = monte_carlo(eta, peso, hb, n=N_INDIVIDUAL, seed=7, t_max_s=T_MAX_MC, dt=DT_MC, use_cc=True)

    fasce.append(nome)
    obs_mean.append(o)
    obs_sd.append(sd)
    gm_lo.append(np.nanpercentile(gmeans, 2.5))
    gm_hi.append(np.nanpercentile(gmeans, 97.5))
    gm_mean.append(np.nanmean(gmeans))
    camp.append(individual)

    print(f"{ETA_EN[nome]:8s} predicted mean={np.nanmean(gmeans):6.1f}s  95% CI group mean [{gm_lo[-1]:.1f}, {gm_hi[-1]:.1f}]  observed={o:.1f}s")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

x = np.arange(len(fasce))
obs_mean = np.array(obs_mean)
obs_sd = np.array(obs_sd)
gm_lo = np.array(gm_lo)
gm_hi = np.array(gm_hi)
gm_mean = np.array(gm_mean)

ax1.fill_between(x, gm_lo, gm_hi, alpha=0.25, label="95% CI of predicted mean (n=10)")
ax1.plot(x, gm_mean, "o-", label="predicted mean")
ax1.errorbar(x, obs_mean, yerr=obs_sd, fmt="s", capsize=4, label="observed (mean$\\pm$SD)")
ax1.set_xticks(x)
ax1.set_xticklabels([ETA_EN[f] for f in fasce], rotation=20, fontsize=8)
ax1.set_ylabel("T$_{90}$ (s)")
ax1.set_title("Rigorous test: observed mean vs CI of the group mean")
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

parts = ax2.violinplot(camp, positions=x, showmedians=True, widths=0.8)
for pc in parts["bodies"]:
    pc.set_alpha(0.32)
ax2.errorbar(x, obs_mean, yerr=obs_sd, fmt="s", capsize=4, label="observed", zorder=5)
ax2.set_xticks(x)
ax2.set_xticklabels([ETA_EN[f] for f in fasce], rotation=20, fontsize=8)
ax2.set_ylabel("T$_{90}$ (s)")
ax2.set_title("Individual dispersion (virtual patients) vs observed")
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("figure3_montecarlo.png", dpi=300, bbox_inches="tight")
plt.show()

## Optional ipywidgets interface

This cell works only if `ipywidgets` is installed/enabled in your Jupyter environment.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    age = widgets.FloatSlider(value=3.5, min=0.0, max=18.0, step=0.1, description="Age")
    weight = widgets.FloatSlider(value=15.0, min=3.0, max=90.0, step=0.5, description="Weight")
    fao2 = widgets.FloatSlider(value=0.90, min=0.21, max=0.98, step=0.01, description="FAO2")
    shunt = widgets.FloatSlider(value=0.05, min=0.0, max=0.30, step=0.01, description="Shunt")
    patent = widgets.Checkbox(value=False, description="Airway patent")
    airway_o2 = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.01, description="Airway O2")
    use_cc_w = widgets.Checkbox(value=True, description="Closing capacity")

    out = widgets.Output()

    def update(*args):
        with out:
            clear_output(wait=True)
            p = costruisci(
                "widget scenario",
                eta=age.value,
                peso=weight.value,
                fao2=fao2.value,
                airway_pervio=patent.value,
                f_aw_o2=airway_o2.value,
                shunt=shunt.value,
                use_cc=use_cc_w.value
            )
            t, spo2, _ = simula(p, t_max_s=1200)
            t90 = tempo_a_soglia(t, spo2)
            plt.figure(figsize=(7, 4))
            plt.plot(t/60, spo2, lw=2)
            plt.axhline(90, ls="--", color="gray")
            if t90:
                plt.scatter([t90/60], [90])
                plt.title(f"T90 = {t90:.1f} s")
            else:
                plt.title("No T90 within simulated window")
            plt.xlabel("Apnoea time (min)")
            plt.ylabel("SpO$_2$ (%)")
            plt.ylim(60, 101)
            plt.grid(alpha=0.3)
            plt.show()

    for w in [age, weight, fao2, shunt, patent, airway_o2, use_cc_w]:
        w.observe(update, names="value")

    display(widgets.VBox([age, weight, fao2, shunt, patent, airway_o2, use_cc_w, out]))
    update()

except Exception as e:
    print("ipywidgets not available or not enabled:", e)